In [ ]:
import os, json
import re, ast
import torch, torch.nn as nn, torch.utils.data, torch.nn.functional as F
import math

from google.colab import drive
from collections import Counter
from torch.utils.data import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# I. Dataset

## 1. Reading original dataset

In [ ]:
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Section 34 - Build a Chatbot with Transformer"

Mounted at /content/drive
/content/drive/MyDrive/Section 34 - Build a Chatbot with Transformer


In [ ]:
print("📂 Current directory:", os.getcwd())
!apt install tree -y
!tree -L 2 -a --dirsfirst

📂 Current directory: /content/drive/MyDrive/Section 34 - Build a Chatbot with Transformer
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 38 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (42.7 kB/s)
Selecting previously unselected package tree.
(Reading database ... 126718 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
.
├── cornell movie-dialogs corpus
│   ├── chameleons.pdf
│   ├── .DS_Store
│   ├── movie_characters_metadata.txt
│   ├── movie_conversations.txt
│   ├── movie_lines.txt
│   ├── movie_titles_metada

In [ ]:
corpus_movie_conv = 'cornell movie-dialogs corpus/movie_conversations.txt'
corpus_movie_lines = 'cornell movie-dialogs corpus/movie_lines.txt'

In [ ]:
with open(corpus_movie_conv, 'r', encoding='utf-8', errors='ignore') as c:  conv = c.readlines()
with open(corpus_movie_lines, 'r', encoding='utf-8', errors='ignore') as l: lines = l.readlines()
# 'r' represents for reading file
# conv and lines are lists

print(conv[0], lines[0])

u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L194', 'L195', 'L196', 'L197']
 L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!



## 2. Create encoded dataset

In [ ]:
lines_dic = {}
for line in lines:
    objects = line.split(" +++$+++ ")
    lines_dic[objects[0]] = objects[-1]

# Print a element of lines_dic
print(next(iter(lines_dic.items())))

('L1045', 'They do not!\n')


In [ ]:
def remove_punc(string):
    return re.sub(r'[^A-Za-z0-9\s]', '', string).lower()

In [ ]:
max_len = 25
pairs = []

for c in conv:
    curr_idxs = ast.literal_eval(c.split(" +++$+++ ")[-1])

    for i in range(len(curr_idxs)):
        qa_pairs = []

        if i == len(curr_idxs) - 1: break

        first =  remove_punc(lines_dic[curr_idxs[i]].strip())
        second = remove_punc(lines_dic[curr_idxs[i+1]].strip())
        qa_pairs.append(first.split()[:max_len])
        qa_pairs.append(second.split()[:max_len])
        pairs.append(qa_pairs)

pairs[0]

[['can',
  'we',
  'make',
  'this',
  'quick',
  'roxanne',
  'korrine',
  'and',
  'andrew',
  'barrett',
  'are',
  'having',
  'an',
  'incredibly',
  'horrendous',
  'public',
  'break',
  'up',
  'on',
  'the',
  'quad',
  'again'],
 ['well',
  'i',
  'thought',
  'wed',
  'start',
  'with',
  'pronunciation',
  'if',
  'thats',
  'okay',
  'with',
  'you']]

In [ ]:
word_freq = Counter()
for pair in pairs:
    word_freq.update(pair[0])
    word_freq.update(pair[1])

In [ ]:
min_word_freq = 5
words = [w for w in word_freq.keys() if word_freq[w] > min_word_freq]
word_map = {k: v + 1 for v, k in enumerate(words)}
word_map['<unk>'] = len(word_map) + 1
word_map['<start>'] = len(word_map) + 1
word_map['<end>'] = len(word_map) + 1
word_map['<pad>'] = 0

print("Total words are: {}".format(len(word_map)))

Total words are: 18188


In [ ]:
with open('WORDMAP_corpus.json', 'w') as j: json.dump(word_map, j)

In [ ]:
with open('WORDMAP_corpus.json', 'r') as j: word_map = json.load(j)

print("'can': ", word_map["can"])
print("'<pad>': ", word_map["<pad>"])
print("'<end>': ", word_map["<end>"])

'can':  1
'<pad>':  0
'<end>':  18187


In [ ]:
def encode_question(words, word_map):
    enc_c = [word_map.get(word, word_map['<unk>']) for word in words] + [word_map['<pad>']] * (max_len - len(words))
    return enc_c

def encode_reply(words, word_map):
    enc_c = [word_map['<start>']] + [word_map.get(word, word_map['<unk>']) for word in words] + \
    [word_map['<end>']] + [word_map['<pad>']] * (max_len - len(words))
    return enc_c

In [ ]:
pairs_encoded = []
for pair in pairs:
    qus = encode_question(pair[0], word_map)
    ans = encode_reply(pair[1], word_map)
    pairs_encoded.append([qus, ans])

with open('pairs_encoded.json', 'w') as p: json.dump(pairs_encoded, p)

# II. Data loading and Masking

## 1. Loading data

In [ ]:
class Dataset(Dataset):

    def __init__(self):

        self.pairs = json.load(open('pairs_encoded.json'))
        self.dataset_size = len(self.pairs)

    def __getitem__(self, i):

        question = torch.LongTensor(self.pairs[i][0])
        reply = torch.LongTensor(self.pairs[i][1])

        return question, reply

    def __len__(self):
        return self.dataset_size

In [ ]:
train_loader = torch.utils.data.DataLoader(Dataset(),
                                           batch_size = 100,
                                           shuffle=True,
                                           pin_memory=True)

In [ ]:
print(next(iter(train_loader)))

[tensor([[   25,    54,   908,  ...,     0,     0,     0],
        [   28,  1986,  1777,  ...,     0,     0,     0],
        [11079,     0,     0,  ...,     0,     0,     0],
        ...,
        [15836,   461,  2440,  ...,     0,     0,     0],
        [   26,   997,    26,  ...,     0,     0,     0],
        [  173,    20,   276,  ...,     0,     0,     0]]), tensor([[18186,   689,    30,  ...,     0,     0,     0],
        [18186,    56,    27,  ...,     0,     0,     0],
        [18186,    57, 15212,  ...,     0,     0,     0],
        ...,
        [18186,     2,    60,  ...,     0,     0,     0],
        [18186,  1740,   344,  ...,     0,     0,     0],
        [18186, 18185,    29,  ...,     0,     0,     0]])]


## 2. Mask creating

In [ ]:
def create_masks(question, reply_input, reply_target):
    def subsequent_mask(size):
        mask = torch.triu(torch.ones(size, size)).transpose(0, 1).type(dtype=torch.uint8)
        return mask.unsqueeze(0)

    """
    Create masks for Transformer (encoder-decoder) during seq2seq training.

    ___ Inputs:
    question       : Tensor[int], shape (batch_size, max_words_q)
                     -> Batch of encoder input sequences (questions), padded with 0.
    reply_input    : Tensor[int], shape (batch_size, max_words_r)
                     -> Batch of decoder input sequences (<sos> ...), padded with 0.
    reply_target   : Tensor[int], shape (batch_size, max_words_r)
                     -> Batch of target sequences (... <eos>), padded with 0.

    ___ Outputs:
    question_mask      : Tensor[bool], shape (batch_size, 1, 1, max_words_q) -> Encoder mask: True for valid tokens, False for padding.
    reply_input_mask   : Tensor[bool], shape (batch_size, 1, max_words_r, max_words_r)
                        -> Decoder self-attention mask: combine padding mask (ignore tokens == 0) and subsequent mask (ignore future tokens)
    reply_target_mask  : Tensor[bool], shape (batch_size, max_words_r) -> Mask for computing loss: ignores padding tokens.
    """
    question_mask = question!=0
    question_mask = question_mask.to(device)
    question_mask = question_mask.unsqueeze(1).unsqueeze(1)         # (batch_size, 1, 1, max_words)

    reply_input_mask = reply_input!=0
    reply_input_mask = reply_input_mask.unsqueeze(1)                # (batch_size, 1, max_words)
    reply_input_mask = reply_input_mask & subsequent_mask(reply_input.size(-1)).type_as(reply_input_mask.data)
    reply_input_mask = reply_input_mask.unsqueeze(1)                # (batch_size, 1, max_words, max_words)

    reply_target_mask = reply_target!=0                             # (batch_size, max_words)

    return question_mask, reply_input_mask, reply_target_mask

# III. Model

## 1. Elements in a Transformer Block: Embedding, Multi-head attention and Feed forward

In [ ]:
class Embeddings(nn.Module):
    """
    Implements embeddings of the words and adds their positional encodings.
    """
    def __init__(self, vocab_size, d_model, max_len = 50):
        super(Embeddings, self).__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pe = self.create_positinal_encoding(max_len, self.d_model)
        self.dropout = nn.Dropout(0.1)

    def create_positinal_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model).to(device)
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i)/d_model)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * (i + 1))/d_model)))
        pe = pe.unsqueeze(0)   # Unsqueeze for batch size -> [1, max_len, d_model]
        return pe

    def forward(self, encoded_words):
        embedding = self.embed(encoded_words) * math.sqrt(self.d_model) # [batch_size, seq_len, d_model]
        embedding += self.pe[:, :embedding.size(1)]                     # Add positional encoding: broadcast pe (1, max_len, d_model) -> [batch_size, seq_len, d_model]
        embedding = self.dropout(embedding)
        return embedding                                                # [batch_size, seq_len, d_model]

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, heads, d_model):
        super(MultiHeadAttention, self).__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.heads = heads
        self.dropout = nn.Dropout(0.1)
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.concat = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask):
        """
        query, key, value of shape: (batch_size, max_len, 512)
        mask of shape: (batch_size, 1, 1, max_words)
        """
        # (batch_size, max_len, d_model)
        query = self.query(query)
        key = self.key(key)
        value = self.value(value)

        # (batch_size, max_len, d_model) --> (batch_size, max_len, h, d_k) --> (batch_size, h, max_len, d_k)
        query = query.view(query.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        key = key.view(key.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        value = value.view(value.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)

        # (batch_size, h, max_len, d_k) matmul (batch_size, h, d_k, max_len) --> (batch_size, h, max_len, max_len)
        scores = torch.matmul(query, key.permute(0,1,3,2)) / math.sqrt(query.size(-1))
        scores = scores.masked_fill(mask == 0, -1e9)    # (batch_size, h, max_len, max_len)

        weights = F.softmax(scores, dim = -1)           # (batch_size, h, max_len, max_len)
        weights = self.dropout(weights)
        # (batch_size, h, max_len, max_len) matmul (batch_size, h, max_len, d_k) --> (batch_size, h, max_len, d_k)
        context = torch.matmul(weights, value)
        # (batch_size, h, max_len, d_k) --> (batch_size, max_len, h, d_k) --> (batch_size, max_len, h * d_k)
        context = context.permute(0,2,1,3).contiguous().view(context.shape[0], -1, self.heads * self.d_k)
        # (batch_size, max_len, h * d_k)
        interacted = self.concat(context)
        return interacted

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, d_model, middle_dim = 2048):
        super(FeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, middle_dim)
        self.fc2 = nn.Linear(middle_dim, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        out = F.relu(self.fc1(x))
        out = self.fc2(self.dropout(out))
        return out

## 2. Encoding layer

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads):
        super(EncoderLayer, self).__init__()
        self.layernorm = nn.LayerNorm(d_model)
        self.self_multihead = MultiHeadAttention(heads, d_model)
        self.feed_forward = FeedForward(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, embeddings, mask):
        interacted = self.dropout(self.self_multihead(embeddings, embeddings, embeddings, mask))    # (batch_size, max_len, d_model)
        interacted = self.layernorm(interacted + embeddings)
        feed_forward_out = self.dropout(self.feed_forward(interacted))
        encoded = self.layernorm(feed_forward_out + interacted)
        return encoded

## 3. Decoding layer

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads):
        super(DecoderLayer, self).__init__()
        self.layernorm = nn.LayerNorm(d_model)
        self.self_multihead = MultiHeadAttention(heads, d_model)
        self.src_multihead = MultiHeadAttention(heads, d_model)
        self.feed_forward = FeedForward(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, embeddings, encoded, src_mask, target_mask):
        query = self.dropout(self.self_multihead(embeddings, embeddings, embeddings, target_mask))
        query = self.layernorm(query + embeddings)
        interacted = self.dropout(self.src_multihead(query, encoded, encoded, src_mask))
        interacted = self.layernorm(interacted + query)
        feed_forward_out = self.dropout(self.feed_forward(interacted))
        decoded = self.layernorm(feed_forward_out + interacted)
        return decoded

## 4. Compile to a Transformer Class

In [ ]:
class Transformer(nn.Module):

    def __init__(self, d_model, heads, num_layers, word_map):
        super(Transformer, self).__init__()

        self.d_model = d_model
        self.vocab_size = len(word_map)
        self.embed = Embeddings(self.vocab_size, d_model)
        self.encoder = nn.ModuleList([EncoderLayer(d_model, heads) for _ in range(num_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, heads) for _ in range(num_layers)])
        self.logit = nn.Linear(d_model, self.vocab_size)

    def encode(self, src_words, src_mask):
        src_embeddings = self.embed(src_words)
        for layer in self.encoder:
            src_embeddings = layer(src_embeddings, src_mask)
        return src_embeddings

    def decode(self, target_words, target_mask, src_embeddings, src_mask):
        tgt_embeddings = self.embed(target_words)
        for layer in self.decoder:
            tgt_embeddings = layer(tgt_embeddings, src_embeddings, src_mask, target_mask)
        return tgt_embeddings

    def forward(self, src_words, src_mask, target_words, target_mask):
        encoded = self.encode(src_words, src_mask)
        decoded = self.decode(target_words, target_mask, encoded, src_mask)
        out = F.log_softmax(self.logit(decoded), dim = 2)
        return out # (batch_size,tgt_len,vocab_size)

# IV. Training and evaluating function

## 1. Adam optimizer

In [ ]:
class AdamWarmUp:
    def __init__(self, model_size, warmup_steps, optimizer):
        self.model_size = model_size
        self.warmup_steps = warmup_steps
        self.optimizer = optimizer
        self.current_step = 0
        self.lr = 0

    def get_lr(self):
        return self.model_size ** (-0.5) * min(self.current_step ** (-0.5), self.current_step * self.warmup_steps ** (-1.5))

    def step(self):
        self.current_step += 1
        lr = self.get_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        self.lr = lr
        # Update weights
        self.optimizer.step()

## 2. KL Loss

In [ ]:
class LossWithLS(nn.Module):
    def __init__(self, size, smooth):
        super(LossWithLS, self).__init__()
        self.criterion = nn.KLDivLoss(size_average=False, reduce = False)
        self.confidence = 1-smooth
        self.smooth = smooth
        self.size = size

    def forward(self, prediction, target, mask):
        """
        prediction of shape: (batch_size, max_words, vocab_size)
        target and mask of shape: (batch_size, max_words)
        """
        prediction = prediction.view(-1, prediction.size(-1))
        # -> (batch_size * max_words, vocab_size)
        target = target.contiguous().view(-1)
        # -> (batch_size * max_words)

        mask = mask.float()
        mask = mask.view(-1)
        # -> (batch_size * max_words)

        labels = prediction.data.clone()
        labels.fill_(self.smooth / (self.size - 1))
        labels.scatter_(1, target.data.unsqueeze(1), self.confidence)
        # -> (batch_size * max_words, vocab_size)

        loss = self.criterion(prediction, labels)
        # -> (batch_size * max_words, vocab_size)
        loss = (loss.sum(1) * mask).sum() / mask.sum()
        return loss

In [ ]:
d_model = 512
heads = 8
num_layers = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 10

with open('WORDMAP_corpus.json', 'r') as j:
    word_map = json.load(j)

transformer = Transformer(d_model = d_model, heads = heads, num_layers = num_layers, word_map = word_map)
transformer = transformer.to(device)
adam_optimizer = torch.optim.Adam(transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
transformer_optimizer = AdamWarmUp(model_size = d_model, warmup_steps = 4000, optimizer = adam_optimizer)
criterion = LossWithLS(len(word_map), 0.1)

/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:51: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


## 3. Training function

In [ ]:
def train(train_loader, transformer, criterion, epoch):

    transformer.train()
    sum_loss = 0
    count = 0

    for i, (question, reply) in enumerate(train_loader):

        samples = question.shape[0]

        # Move to device
        question = question.to(device)
        reply = reply.to(device)

        # Prepare Target Data
        reply_input = reply[:, :-1]
        reply_target = reply[:, 1:]

        # Create mask and add dimensions
        question_mask, reply_input_mask, reply_target_mask = create_masks(question, reply_input, reply_target)

        # Get the transformer outputs
        out = transformer(question, question_mask, reply_input, reply_input_mask)

        # Compute the loss
        loss = criterion(out, reply_target, reply_target_mask)

        # Backprop
        transformer_optimizer.optimizer.zero_grad()
        loss.backward()
        transformer_optimizer.step()

        sum_loss += loss.item() * samples
        count += samples

        if i % 100 == 0:
            print("Epoch [{}][{}/{}]\tLoss: {:.3f}".format(epoch, i, len(train_loader), sum_loss/count))

## 4. Evaluating function

In [ ]:
def evaluate(transformer, question, question_mask, max_len, word_map):
    """
    Performs Greedy Decoding with a batch size of 1
    """
    rev_word_map = {v: k for k, v in word_map.items()}
    transformer.eval()
    start_token = word_map['<start>']
    encoded = transformer.encode(question, question_mask)
    words = torch.LongTensor([[start_token]]).to(device)

    for step in range(max_len - 1):
        size = words.shape[1]
        target_mask = torch.triu(torch.ones(size, size)).transpose(0, 1).type(dtype=torch.uint8)
        target_mask = target_mask.to(device).unsqueeze(0).unsqueeze(0)
        decoded = transformer.decode(words, target_mask, encoded, question_mask)
        predictions = transformer.logit(decoded[:, -1])
        _, next_word = torch.max(predictions, dim = 1)
        next_word = next_word.item()
        if next_word == word_map['<end>']:
            break
        words = torch.cat([words, torch.LongTensor([[next_word]]).to(device)], dim = 1)   # (1,step+2)

    # Construct Sentence
    if words.dim() == 2:
        words = words.squeeze(0)
        words = words.tolist()

    sen_idx = [w for w in words if w not in {word_map['<start>']}]
    sentence = ' '.join([rev_word_map[sen_idx[k]] for k in range(len(sen_idx))])

    return sentence

# IV. Training and evaluating session

In [ ]:
for epoch in range(epochs):

    train(train_loader, transformer, criterion, epoch)

    state = {'epoch': epoch, 'transformer': transformer, 'transformer_optimizer': transformer_optimizer}
    torch.save(state, 'checkpoint_' + str(epoch) + '.pth.tar')

Epoch [0][0/2217]	Loss: 8.652
Epoch [0][100/2217]	Loss: 7.916
Epoch [0][200/2217]	Loss: 7.201
Epoch [0][300/2217]	Loss: 6.664
Epoch [0][400/2217]	Loss: 6.306
Epoch [0][500/2217]	Loss: 6.057
Epoch [0][600/2217]	Loss: 5.872
Epoch [0][700/2217]	Loss: 5.723
Epoch [0][800/2217]	Loss: 5.605
Epoch [0][900/2217]	Loss: 5.507
Epoch [0][1000/2217]	Loss: 5.424
Epoch [0][1100/2217]	Loss: 5.355
Epoch [0][1200/2217]	Loss: 5.294
Epoch [0][1300/2217]	Loss: 5.240
Epoch [0][1400/2217]	Loss: 5.192
Epoch [0][1500/2217]	Loss: 5.151
Epoch [0][1600/2217]	Loss: 5.112
Epoch [0][1700/2217]	Loss: 5.077
Epoch [0][1800/2217]	Loss: 5.046
Epoch [0][1900/2217]	Loss: 5.018
Epoch [0][2000/2217]	Loss: 4.991
Epoch [0][2100/2217]	Loss: 4.966
Epoch [0][2200/2217]	Loss: 4.944
Epoch [1][0/2217]	Loss: 4.514
Epoch [1][100/2217]	Loss: 4.420
Epoch [1][200/2217]	Loss: 4.417
Epoch [1][300/2217]	Loss: 4.413
Epoch [1][400/2217]	Loss: 4.414
Epoch [1][500/2217]	Loss: 4.410
Epoch [1][600/2217]	Loss: 4.409
Epoch [1][700/2217]	Loss: 4.410

KeyboardInterrupt: 

In [ ]:
checkpoint = torch.load('checkpoint_1.pth.tar', weights_only=False)
transformer = checkpoint['transformer']

In [ ]:
while(1):
    question = input("Question: ")
    if question == 'quit':
        break
    max_len = input("Maximum Reply Length: ")
    enc_qus = [word_map.get(word, word_map['<unk>']) for word in question.split()]
    question = torch.LongTensor(enc_qus).to(device).unsqueeze(0)
    question_mask = (question!=0).to(device).unsqueeze(1).unsqueeze(1)
    sentence = evaluate(transformer, question, question_mask, int(max_len), word_map)
    print(sentence)

Question: hello
Maximum Reply Length: 25
i dont know
Question: why
Maximum Reply Length: 25
i dont know
Question: what are you doing
Maximum Reply Length: 6
i dont know
Question: haha
Maximum Reply Length: 5
i dont know
Question: quit
